# 🩸 Blood Biomarker Detection & Severity Analysis

This notebook implements a full end-to-end pipeline:
1. **Generates synthetic blood sample data** — Hemoglobin, Glucose, Albumin
2. **Simulates colorimetric strip image color changes** — RGB values encode biomarker concentrations
3. **ML Model 1** — Predicts biomarker readings from image color features (Multi-Output Random Forest Regressor)
4. **ML Model 2** — Classifies overall health severity from biomarker readings (Gradient Boosting Classifier)

| Biomarker | Normal Range | Unit |
|-----------|-------------|------|
| Hemoglobin | 12.0 – 17.5 | g/dL |
| Glucose | 70 – 140 | mg/dL |
| Albumin | 3.5 – 5.0 | g/dL |

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, classification_report, confusion_matrix
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)
print('All packages imported successfully!')

## 📊 Step 1 — Generate Synthetic Biomarker Data

In [ ]:
N = 1500
rng = np.random.default_rng(42)
hemoglobin = np.clip(rng.normal(13.5, 2.5, N), 4, 22)
glucose    = np.clip(rng.lognormal(4.9, 0.35, N), 40, 500)
albumin    = np.clip(rng.normal(4.1, 0.6, N), 1.5, 6.0)
R = np.clip(10*hemoglobin + 0.05*glucose - 5*albumin  + rng.normal(0, 8, N),  0, 255)
G = np.clip(-0.3*hemoglobin + 0.45*glucose + 2*albumin + rng.normal(0, 10, N), 0, 255)
B = np.clip(3*hemoglobin - 0.1*glucose + 20*albumin  + rng.normal(0, 7, N),  0, 255)
brightness = (R + G + B) / 3
rg_ratio   = R / (G + 1)
rb_ratio   = R / (B + 1)
max_rgb    = np.maximum.reduce([R, G, B])
min_rgb    = np.minimum.reduce([R, G, B])
saturation = (max_rgb - min_rgb) / (max_rgb + 1)
df = pd.DataFrame({
    'hemoglobin': hemoglobin, 'glucose': glucose, 'albumin': albumin,
    'R': R, 'G': G, 'B': B,
    'brightness': brightness, 'rg_ratio': rg_ratio,
    'rb_ratio': rb_ratio, 'saturation': saturation
})
print(f'Dataset shape: {df.shape}')
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
specs = [('hemoglobin','Hemoglobin (g/dL)','#e74c3c',(12.0,17.5)),
         ('glucose','Glucose (mg/dL)','#f39c12',(70,140)),
         ('albumin','Albumin (g/dL)','#3498db',(3.5,5.0))]
for ax, (col, label, color, (lo, hi)) in zip(axes, specs):
    ax.hist(df[col], bins=40, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(lo, color='green', linestyle='--', linewidth=1.8, label=f'Normal low ({lo})')
    ax.axvline(hi, color='darkgreen', linestyle='--', linewidth=1.8, label=f'Normal high ({hi})')
    ax.set_title(label, fontweight='bold'); ax.set_xlabel(label); ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.suptitle('Biomarker Distributions with Normal Range Markers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 🖼️ Step 2 — Simulated Colorimetric Strip Images

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(16, 5))
axes = axes.flatten()
sample_idx = np.linspace(0, N-1, 12, dtype=int)
for i, idx in enumerate(sample_idx):
    r, g, b = int(df.loc[idx,'R']), int(df.loc[idx,'G']), int(df.loc[idx,'B'])
    swatch = np.full((60, 80, 3), [r, g, b], dtype=np.uint8)
    axes[i].imshow(swatch)
    axes[i].set_title(f'Hb={df.loc[idx,"hemoglobin"]:.1f}\nGl={df.loc[idx,"glucose"]:.0f}\nAl={df.loc[idx,"albumin"]:.1f}', fontsize=7.5)
    axes[i].axis('off')
plt.suptitle('Simulated Colorimetric Strip Swatches (color encodes biomarker concentrations)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
from scipy.stats import pearsonr
channels = ['R','G','B']; biomarkers = ['hemoglobin','glucose','albumin']
colors_map = {'R':'#e74c3c','G':'#27ae60','B':'#2980b9'}
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for row, bio in enumerate(biomarkers):
    for col, ch in enumerate(channels):
        ax = axes[row][col]
        ax.scatter(df[ch], df[bio], alpha=0.25, s=8, color=colors_map[ch])
        r_val, _ = pearsonr(df[ch], df[bio])
        ax.set_xlabel(f'{ch} channel', fontsize=9); ax.set_ylabel(bio.capitalize(), fontsize=9)
        ax.set_title(f'{ch} vs {bio.capitalize()}  r={r_val:.3f}', fontsize=9, fontweight='bold')
plt.suptitle('RGB Channel vs Biomarker Correlations', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 🤖 Step 3 — ML Model 1: Predict Biomarker Levels from Image Color

In [ ]:
FEATURES = ['R','G','B','brightness','rg_ratio','rb_ratio','saturation']
TARGETS   = ['hemoglobin','glucose','albumin']
X = df[FEATURES].values; y = df[TARGETS].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model1 = MultiOutputRegressor(RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1))
model1.fit(X_train, y_train)
y_pred = model1.predict(X_test)
print('Model 1 — Performance on Test Set'); print('='*45)
units = ['g/dL','mg/dL','g/dL']
for i, (tgt, unit) in enumerate(zip(TARGETS, units)):
    mae = mean_absolute_error(y_test[:,i], y_pred[:,i])
    r2  = r2_score(y_test[:,i], y_pred[:,i])
    print(f'  {tgt.capitalize():12s}  MAE={mae:6.3f} {unit:6s}  R2={r2:.4f}')
y_full = model1.predict(X)
df['pred_hemoglobin'] = y_full[:,0]; df['pred_glucose'] = y_full[:,1]; df['pred_albumin'] = y_full[:,2]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_bio = ['#e74c3c','#f39c12','#3498db']
for i, (tgt, color) in enumerate(zip(TARGETS, colors_bio)):
    ax = axes[i]
    ax.scatter(y_test[:,i], y_pred[:,i], alpha=0.4, s=15, color=color)
    lims = [min(y_test[:,i].min(),y_pred[:,i].min()), max(y_test[:,i].max(),y_pred[:,i].max())]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect fit')
    r2 = r2_score(y_test[:,i],y_pred[:,i]); mae = mean_absolute_error(y_test[:,i],y_pred[:,i])
    ax.set_xlabel(f'Actual {tgt}',fontsize=10); ax.set_ylabel(f'Predicted {tgt}',fontsize=10)
    ax.set_title(f'{tgt.capitalize()}\nR2={r2:.4f}  MAE={mae:.3f}', fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle('Model 1 — Predicted vs Actual Biomarker Values', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_bio = ['#e74c3c','#f39c12','#3498db']
for i, (tgt, color) in enumerate(zip(TARGETS, colors_bio)):
    importances = model1.estimators_[i].feature_importances_
    order = np.argsort(importances)[::-1]
    axes[i].bar([FEATURES[j] for j in order], importances[order], color=color, alpha=0.8, edgecolor='white')
    axes[i].set_title(f'Feature Importance — {tgt.capitalize()}', fontweight='bold')
    axes[i].set_ylabel('Importance'); axes[i].tick_params(axis='x', rotation=35)
plt.suptitle('Model 1 — Feature Importances per Biomarker', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## ⚠️ Step 4 — Severity Labeling

In [ ]:
def label_hemoglobin(v):
    if v < 7: return 'Severe Anaemia'
    if v < 10: return 'Moderate Anaemia'
    if v < 12: return 'Mild Anaemia'
    if v <= 17.5: return 'Normal'
    return 'Polycythemia'

def label_glucose(v):
    if v < 54: return 'Severe Hypoglycaemia'
    if v < 70: return 'Hypoglycaemia'
    if v <= 99: return 'Normal'
    if v <= 125: return 'Pre-Diabetic'
    if v <= 200: return 'Diabetic'
    return 'Severely Diabetic'

def label_albumin(v):
    if v < 2: return 'Severe Hypoalbuminaemia'
    if v < 3.5: return 'Hypoalbuminaemia'
    if v <= 5: return 'Normal'
    return 'Hyperalbuminaemia'

SEVERE_SET   = {'Severe Anaemia','Severe Hypoglycaemia','Severe Hypoalbuminaemia','Severely Diabetic'}
HIGH_SET     = {'Polycythemia','Diabetic','Moderate Anaemia'}
MODERATE_SET = {'Mild Anaemia','Pre-Diabetic','Hypoglycaemia','Hypoalbuminaemia','Hyperalbuminaemia'}

def overall_severity(hb_lbl, gl_lbl, al_lbl):
    labels = {hb_lbl, gl_lbl, al_lbl}
    if labels & SEVERE_SET: return 'Critical'
    if labels & HIGH_SET: return 'High'
    if labels & MODERATE_SET: return 'Moderate'
    return 'Normal'

df['hb_label'] = df['pred_hemoglobin'].apply(label_hemoglobin)
df['gl_label'] = df['pred_glucose'].apply(label_glucose)
df['al_label'] = df['pred_albumin'].apply(label_albumin)
df['severity'] = df.apply(lambda r: overall_severity(r['hb_label'],r['gl_label'],r['al_label']), axis=1)
print('Severity distribution:'); print(df['severity'].value_counts())

In [ ]:
severity_counts = df['severity'].value_counts()
sev_colors  = {'Critical':'#e74c3c','High':'#e67e22','Moderate':'#f1c40f','Normal':'#2ecc71'}
plot_colors = [sev_colors[s] for s in severity_counts.index]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.pie(severity_counts.values, labels=severity_counts.index, autopct='%1.1f%%',
        colors=plot_colors, startangle=140, pctdistance=0.82,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title('Severity Distribution (Pie)', fontweight='bold', fontsize=13)
bars = ax2.bar(severity_counts.index, severity_counts.values, color=plot_colors, edgecolor='white')
for bar, val in zip(bars, severity_counts.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val), ha='center', fontweight='bold')
ax2.set_title('Severity Distribution (Bar)', fontweight='bold', fontsize=13)
ax2.set_ylabel('Patient Count'); ax2.set_xlabel('Severity Level')
plt.suptitle('Overall Health Severity Distribution', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 🤖 Step 5 — ML Model 2: Severity Classifier

In [ ]:
le = LabelEncoder()
y_sev = le.fit_transform(df['severity'])
X2 = df[['pred_hemoglobin','pred_glucose','pred_albumin']].values
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y_sev, test_size=0.2, stratify=y_sev, random_state=42)
model2 = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.08, random_state=42)
model2.fit(X2_train, y2_train)
y2_pred = model2.predict(X2_test)
print('Model 2 — Severity Classification Report'); print('='*50)
print(classification_report(y2_test, y2_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y2_test, y2_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_xlabel('Predicted Severity', fontsize=12); ax.set_ylabel('Actual Severity', fontsize=12)
ax.set_title('Model 2 — Confusion Matrix (Severity Classifier)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
feat2 = ['Predicted Hemoglobin','Predicted Glucose','Predicted Albumin']
imp2  = model2.feature_importances_; order2 = np.argsort(imp2)[::-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([feat2[j] for j in order2], imp2[order2],
       color=['#9b59b6','#1abc9c','#e67e22'], edgecolor='white', linewidth=1.5)
ax.set_title('Model 2 — Feature Importance (Severity Classifier)', fontweight='bold', fontsize=12)
ax.set_ylabel('Importance'); plt.tight_layout(); plt.show()

## 🏥 Step 6 — End-to-End Inference Pipeline

In [ ]:
def full_pipeline(r, g, b, patient_id='Patient'):
    brightness = (r+g+b)/3; rg_ratio=r/(g+1); rb_ratio=r/(b+1)
    max_rgb=max(r,g,b); min_rgb=min(r,g,b); saturation=(max_rgb-min_rgb)/(max_rgb+1)
    features = np.array([[r,g,b,brightness,rg_ratio,rb_ratio,saturation]])
    pred_hb, pred_gl, pred_al = model1.predict(features)[0]
    hb_lbl=label_hemoglobin(pred_hb); gl_lbl=label_glucose(pred_gl); al_lbl=label_albumin(pred_al)
    sev_code  = model2.predict(np.array([[pred_hb,pred_gl,pred_al]]))[0]
    sev_proba = model2.predict_proba(np.array([[pred_hb,pred_gl,pred_al]]))[0]
    severity  = le.inverse_transform([sev_code])[0]
    emoji_map = {'Critical':'[CRITICAL]','High':'[HIGH]','Moderate':'[MODERATE]','Normal':'[NORMAL]'}
    sep = '+' + '='*52 + '+'
    print(f'\n{sep}')
    print(f'|  Blood Analysis Report — {patient_id:<26}|')
    print(sep)
    print(f'|  Input Strip RGB : ({int(r):3d},{int(g):3d},{int(b):3d}){" "*24}|')
    print(f'|  BIOMARKER PREDICTIONS:{" "*30}|')
    print(f'|    Hemoglobin : {pred_hb:6.2f} g/dL   -> {hb_lbl:<22}|')
    print(f'|    Glucose    : {pred_gl:6.1f} mg/dL  -> {gl_lbl:<22}|')
    print(f'|    Albumin    : {pred_al:6.2f} g/dL   -> {al_lbl:<22}|')
    print(f'|  OVERALL SEVERITY: {emoji_map[severity]} {severity:<33}|')
    print(f'|  SEVERITY PROBABILITIES:{" "*29}|')
    for cls, prob in zip(le.classes_, sev_proba):
        bar = '#'*int(prob*20)+'.'*(20-int(prob*20))
        print(f'|    {cls:<10}: [{bar}] {prob*100:5.1f}%              |')
    print(sep)

patients = [(180,90,110,'Alice (Normal)'),(80,200,60,'Bob (High Glucose)'),
            (60,50,45,'Carol (Anaemia)'),(220,240,200,'David (Mixed)')]
for r,g,b,pid in patients:
    full_pipeline(r,g,b,pid)

## 📋 Step 7 — Summary Dashboard

In [ ]:
severity_order = ['Normal','Moderate','High','Critical']
bio_cols = ['pred_hemoglobin','pred_glucose','pred_albumin']
mean_by_sev = (df.groupby('severity')[bio_cols].mean()
               .reindex([s for s in severity_order if s in df['severity'].unique()]))
mean_norm = (mean_by_sev - mean_by_sev.min())/(mean_by_sev.max()-mean_by_sev.min()+1e-9)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(mean_norm, annot=mean_by_sev.round(2), fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, linecolor='white',
            xticklabels=['Hemoglobin\n(g/dL)','Glucose\n(mg/dL)','Albumin\n(g/dL)'], ax=axes[0])
axes[0].set_title('Mean Predicted Biomarker by Severity Group', fontweight='bold')
axes[0].set_ylabel('Severity Group')
sev_present = [s for s in severity_order if s in df['severity'].unique()]
box_colors  = {'Critical':'#e74c3c','High':'#e67e22','Moderate':'#f1c40f','Normal':'#2ecc71'}
data_by_sev = [df.loc[df['severity']==s,'pred_hemoglobin'].values for s in sev_present]
bp = axes[1].boxplot(data_by_sev, labels=sev_present, patch_artist=True, notch=True)
for patch, sev in zip(bp['boxes'], sev_present):
    patch.set_facecolor(box_colors[sev]); patch.set_alpha(0.75)
axes[1].set_title('Predicted Hemoglobin Distribution by Severity', fontweight='bold')
axes[1].set_ylabel('Predicted Hemoglobin (g/dL)'); axes[1].set_xlabel('Severity Group')
plt.suptitle('Summary Dashboard — Biomarker x Severity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print('\n'+'='*60)
print('PIPELINE SUMMARY'); print('='*60)
print('Dataset       : 1,500 synthetic patient samples')
print('Input Features: R, G, B + derived color features (7 total)')
print('MODEL 1 — MultiOutputRegressor(RandomForestRegressor)')
print('  Task: Predict Hemoglobin, Glucose, Albumin from RGB color')
print('SEVERITY LABELING: Rule-based, 4 tiers: Normal->Moderate->High->Critical')
print('MODEL 2 — GradientBoostingClassifier')
print('  Task: Classify severity from predicted biomarkers')
print('INFERENCE: full_pipeline(R, G, B) -> Severity + Biomarkers')
print('='*60)